In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_product_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "ProductId  string "
        #     ",SourceSystem string"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId string"
        #     ",ProductKey  string "
        #     ",ProductCode  string "
        #     ",ProductDescription  string"
        #     ",SubProductCode  int"
        #     ",SubProduct  string"
        #     ",BankingUnitCode  string"
        #     ",InterestUnit  string"
        #     ",InterestFrequency  int"
        #     ",DeliquentInterestUnit  string"
        #     ",DeliquentInterestFrequency  int"
        #     ",InterestDate  date"
        #     ",LastValeurDate  date"
        #     ",NextValeurDate  date"
        #     ",PeriodicIntFlag  string"
        #     ",ShadowFlag  boolean"
        #     ",SyndicatedFlag  boolean"
        #     ",L3869Flag  boolean"
        #     ",OCWFlag  boolean"
        #     ",BondedFlag  boolean"
        #     ",ProductCategory  boolean"
        #     ",BusinessDate  date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Product"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ========================== profits_loan_product ==========================
        profits_loan_product_extra_filters = None

        if profits_loan_product_extra_filters is not None:
            profits_loan_product_extra_filters.append(date_filter)
            profits_loan_product_final_filters = " AND ".join(
                profits_loan_product_extra_filters
            )

        else:
            profits_loan_product_final_filters = date_filter

        profits_loan_product_selected_fields = [
            "product_code",
            "nrm_interest_unit",
            "nrm_interest_freq",
            "ov_interest_unit",
            "ov_interest_freq",
            "nrm_interest_dt",
            "nrm_lst_valeur_dt",
            "nrm_nxt_valeur_dt",
            "periodic_int_flg",
            "businessdate",
        ]

        profits_loan_product = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_product",
            profits_loan_product_final_filters,
            profits_loan_product_selected_fields,
        )

        # ========================== params ==========================
        params_extra_filters = None

        if params_extra_filters is not None:
            params_extra_filters.append(date_filter)
            params_final_filters = " AND ".join(params_extra_filters)

        else:
            params_final_filters = date_filter

        params_selected_fields = [
            "param_type",
            "serial_num",
            "description",
        ]

        params = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_params",
            params_final_filters,
            params_selected_fields,
        )

        params1 = params.filter(params["param_type"] == "PRODUCT")  # filter params
        
        
        # ========================== params_sub (2nd instance) ==========================
        params_sub = (
            params.select("param_type","serial_num", "description")
            .withColumnRenamed("serial_num", "serial_num_sub")
            .withColumnRenamed("description", "subproduct_description")
        )
        params_sub = params_sub.filter(params_sub["param_type"] == "BSECT")  # filter params


        # ========================== loan_details ==========================
        loan_details_extra_filters = None

        if loan_details_extra_filters is not None:
            loan_details_extra_filters.append(date_filter)
            loan_details_final_filters = " AND ".join(loan_details_extra_filters)
        else:
            loan_details_final_filters = date_filter

        loan_details_selected_fields = ["gd_bsect", "prft_product_code"]

        loan_details = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_details",
            loan_details_final_filters,
            loan_details_selected_fields,
        )

        loan_details = (
            loan_details.distinct()
        )  # granularity account_number distinct to eliminate

        # ========================== loan_product_mod ==========================
        loan_product_mod_extra_filters = None

        if loan_product_mod_extra_filters is not None:
            loan_product_mod_extra_filters.append(date_filter)
            loan_product_mod_final_filters = " AND ".join(
                loan_product_mod_extra_filters
            )
        else:
            loan_product_mod_final_filters = date_filter

        loan_product_mod_selected_fields = [
            "shadowflag",
            "prft_product_code",
        ]

        loan_product_mod = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "params_loan_product_mod",
            loan_product_mod_final_filters,
            loan_product_mod_selected_fields,
        )

        # ========================== params_loan_product ==========================
        params_loan_product_extra_filters = None

        if params_loan_product_extra_filters is not None:
            params_loan_product_extra_filters.append(date_filter)
            params_loan_product_final_filters = " AND ".join(
                params_loan_product_extra_filters
            )
        else:
            params_loan_product_final_filters = date_filter

        params_loan_product_selected_fields = [
            "flag",
            "gd_bsect",
            "prft_product_code",
        ]

        params_loan_product = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "params_loan_product",
            params_loan_product_final_filters,
            params_loan_product_selected_fields,
        )

        params_loan_product_synd = params_loan_product.filter(
            params_loan_product["flag"] == "Syndicated"
        )
        params_loan_product_synd = (
            params_loan_product_synd.withColumnRenamed("flag", "flag_synd")
            .withColumnRenamed("gd_bsect", "gd_bsect_synd")
            .withColumnRenamed("prft_product_code", "prft_product_code_synd")
        )

        params_loan_product_locw = params_loan_product.filter(
            params_loan_product["flag"].isin(["L3869", "OCW"])
        )
        params_loan_product_locw = (
            params_loan_product_locw.withColumnRenamed("flag", "flag_locw")
            .withColumnRenamed("gd_bsect", "gd_bsect_locw")
            .withColumnRenamed("prft_product_code", "prft_product_code_locw")
        )

        params_loan_product_bon = params_loan_product.filter(
            params_loan_product["flag"] == "Bonded"
        )
        params_loan_product_bon = (
            params_loan_product_bon.withColumnRenamed("flag", "flag_bon")
            .withColumnRenamed("gd_bsect", "gd_bsect_bon")
            .withColumnRenamed("prft_product_code", "prft_product_code_bon")
        )

        # ============================ join conditions ==========

        profits_loan_product_loan_details_join_condition = (
            profits_loan_product["product_code"] == loan_details["prft_product_code"]
        )

        profits_loan_product_params_join_condition = (
            profits_loan_product["product_code"] == params1["serial_num"]
        )

        loan_details_loan_product_mod_join_condition = (
            loan_details["prft_product_code"] == loan_product_mod["prft_product_code"]
        )

        loan_details_params_loan_product_synd_join_condition = (
            loan_details["gd_bsect"] == params_loan_product_synd["gd_bsect_synd"]
        ) | (
            loan_details["prft_product_code"]
            == params_loan_product_synd["prft_product_code_synd"]
        )

        loan_details_params_loan_product_locw_join_condition = (
            loan_details["gd_bsect"] == params_loan_product_locw["gd_bsect_locw"]
        ) | (
            loan_details["prft_product_code"]
            == params_loan_product_locw["prft_product_code_locw"]
        )

        loan_details_params_loan_product_bon_join_condition = (
            loan_details["gd_bsect"] == params_loan_product_bon["gd_bsect_bon"]
        ) | (
            loan_details["prft_product_code"]
            == params_loan_product_bon["prft_product_code_bon"]
        )

        loan_details_params_sub_join_condition = (
            loan_details["gd_bsect"] == params_sub["serial_num_sub"]
        )
        # ============================ joins ==========

        loan_pr_dtl_table = (
            profits_loan_product.join(
                loan_details, profits_loan_product_loan_details_join_condition, "left"
            )
            .join(params1, profits_loan_product_params_join_condition, "left")
            .join(params_sub, loan_details_params_sub_join_condition, "left")  
            .join(
                loan_product_mod, loan_details_loan_product_mod_join_condition, "left"
            )
            .join(
                params_loan_product_synd,
                loan_details_params_loan_product_synd_join_condition,
                "left",
            )
            .join(
                params_loan_product_locw,
                loan_details_params_loan_product_locw_join_condition,
                "left",
            )
            .join(
                params_loan_product_bon,
                loan_details_params_loan_product_bon_join_condition,
                "left",
            )
        )

        loan_pr_dtl_primary_key_fields = [
            loan_pr_dtl_table.product_code,
            loan_details.gd_bsect,
            lit("profits"),
        ]

        # ============product_profits==============
        product_profits = (
            loan_pr_dtl_table.withColumn(
                "ProductId", sha2(concat_ws(";", *loan_pr_dtl_primary_key_fields), 256)
            )
            .withColumn(
                "ProductKey",
                concat_ws(
                    ";",
                    loan_pr_dtl_table.product_code,
                    loan_details.gd_bsect,
                ),
            )
            .withColumn(
                "ProductCode",
                profits_loan_product.product_code.cast("string"),
            )
            .withColumn(
                "SubProductCode",
                loan_details.gd_bsect.cast("int"),            
            )
            .withColumn(
                "BankingUnitCode",
                loan_details.gd_bsect.cast("string"),
            )
            .withColumn("ProductDescription", params1.description.cast("string"))
            .withColumn("SubProduct", params_sub.subproduct_description.cast("string"))

            .withColumn(
                "SyndicatedFlag",
                when(params_loan_product_synd["flag_synd"].isNotNull(), 1)
                .otherwise(0)
                .cast("boolean"),
            )
            .withColumn(
                "L3869Flag",
                when(params_loan_product_locw["flag_locw"] == "L3869", 1)
                .otherwise(lit(None))
                .cast("boolean"),
            )
            .withColumn(
                "OCWFlag",
                when(params_loan_product_locw["flag_locw"] == "OCW", 1)
                .otherwise(lit(None))
                .cast("boolean"),
            )
            .withColumn(
                "BondedFlag",
                when(params_loan_product_bon["flag_bon"].isNotNull(), 1)
                .otherwise(0)
                .cast("boolean"),
            )
        )

        product_profits = product_profits.select(
            product_profits.ProductId,
            lit("profits").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            product_profits.ProductKey.cast("string").alias("ProductKey"),
            product_profits.ProductCode.cast("string").alias("ProductCode"),
            product_profits.ProductDescription,
            product_profits.SubProductCode.cast("int").alias("SubProductCode"),
            product_profits.SubProduct.cast("string").alias("SubProduct"),
            product_profits.BankingUnitCode.cast("string"),
            product_profits.nrm_interest_unit.alias("InterestUnit"),
            product_profits.nrm_interest_freq.alias("InterestFrequency"),
            product_profits.ov_interest_unit.alias("DeliquentInterestUnit"),
            product_profits.ov_interest_freq.alias("DeliquentInterestFrequency"),
            product_profits.nrm_interest_dt.alias("InterestDate"),
            product_profits.nrm_lst_valeur_dt.alias("LastValeurDate"),
            product_profits.nrm_nxt_valeur_dt.alias("NextValeurDate"),
            product_profits.periodic_int_flg.alias("PeriodicIntFlag"),
            product_profits.shadowflag.cast("boolean").alias("ShadowFlag"),
            product_profits.SyndicatedFlag.cast("boolean").alias("SyndicatedFlag"),
            product_profits.L3869Flag.cast("boolean").alias("L3869Flag"),
            product_profits.OCWFlag.cast("boolean").alias("OCWFlag"),
            product_profits.BondedFlag.cast("boolean").alias("BondedFlag"),
            lit(None).cast("string").alias("ProductCategory"),
            product_profits.businessdate.cast("date").alias("BusinessDate"),
        ).distinct()

        # ================== Append ==================

        product_profits.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_product_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)